# Ingestion Config

The ingestion config controls what the platform extracts from your videos and images during indexing. This is the most important decision you make when creating a knowledge store.

In [ ]:
import os

from twelvelabs import (
    TwelveLabs,
    IngestionConfig,
    EnrichmentConfig_Description,
    EnrichmentConfig_JsonSchema,
    EnrichmentConfigJsonSchemaJsonSchema,
)

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")

client = TwelveLabs(api_key=API_KEY)

## When You Need This

- You want Jockey to focus extraction on specific aspects of your videos and images
- You need structured metadata in a specific shape
- Default extraction doesn't capture what matters for your use case

## Why It Matters

Ingestion config is optional. Jockey works out of the box with default extraction. Configure it when you know what your use case needs — it helps Jockey emphasize the right signals and produce more reliable extraction.

Use this notebook to choose between default extraction, a natural-language description, and JSON Schema.

## Option 1: Natural Language Description

Tell Jockey what to focus on in plain English. Jockey interprets your description to guide extraction.

**Best for:** Exploratory work, when you're not sure exactly what fields you need, or when the extraction shape is hard to express as a schema.

### Example: Surveillance Use Case

In [ ]:
# Surveillance use case — natural language description
surveillance_store = client.knowledge_stores.create(
    name="Parking Lot Monitoring",
    ingestion_config=IngestionConfig(
        enrichment_config=EnrichmentConfig_Description(
            description=(
                "Track individuals across camera views. Focus on physical "
                "descriptions, clothing, direction of movement, timestamps, "
                "and interactions between people."
            )
        )
    ),
)
print(f"Store ID: {surveillance_store.id}")

### Example: Content Library Use Case

In [ ]:
# Content library use case — natural language description
content_store = client.knowledge_stores.create(
    name="Brand Content Library",
    ingestion_config=IngestionConfig(
        enrichment_config=EnrichmentConfig_Description(
            description=(
                "Categorize by visual mood, topic, on-screen talent, product "
                "appearances, and call-to-action type. Note production quality "
                "and format (interview, b-roll, testimonial)."
            )
        )
    ),
)
print(f"Store ID: {content_store.id}")

## Option 2: JSON Schema

Provide a JSON Schema (draft 2020-12) for precise, structured extraction.

**Best for:** Production systems where downstream code expects specific fields, or when you need consistent structure across all indexed videos and images.

### Example: Sports Analysis Use Case

In [ ]:
# Sports analysis use case — JSON schema
# Every property must include a "description" — the platform uses it to guide extraction quality.
SPORTS_SCHEMA = EnrichmentConfigJsonSchemaJsonSchema(
    type="object",
    properties={
        "players": {
            "type": "array",
            "description": "Players identified in the footage",
            "items": {
                "type": "object",
                "properties": {
                    "name_or_number": {"type": "string", "description": "Player name or jersey number"},
                    "team": {"type": "string", "description": "Team the player belongs to"},
                    "position": {"type": "string", "description": "Playing position on the field"},
                },
            },
        },
        "plays": {
            "type": "array",
            "description": "Individual plays or actions during the game",
            "items": {
                "type": "object",
                "properties": {
                    "play_type": {"type": "string", "description": "Type of play such as pass, run, or kick"},
                    "timestamp": {"type": "string", "description": "When the play occurred in the video"},
                    "outcome": {"type": "string", "description": "Result of the play such as complete, incomplete, or turnover"},
                    "players_involved": {
                        "type": "array",
                        "description": "Names or numbers of players involved in the play",
                        "items": {"type": "string"},
                    },
                },
            },
        },
        "score_changes": {
            "type": "array",
            "description": "Scoring events during the game",
            "items": {
                "type": "object",
                "properties": {
                    "timestamp": {"type": "string", "description": "When the score changed in the video"},
                    "team": {"type": "string", "description": "Team that scored"},
                    "new_score": {"type": "string", "description": "Updated score after the event"},
                },
            },
        },
    },
)

sports_store = client.knowledge_stores.create(
    name="Game Film Analysis",
    ingestion_config=IngestionConfig(
        enrichment_config=EnrichmentConfig_JsonSchema(json_schema=SPORTS_SCHEMA)
    ),
)
print(f"Store ID: {sports_store.id}")

## Option 3: No Config (Default)

Omit `ingestion_config` entirely. Jockey uses default extraction.

**Best for:** Getting started, quick prototypes, general-purpose exploration.

In [ ]:
# Default extraction — no ingestion config
default_store = client.knowledge_stores.create(name="Quick Test Store")
print(f"Store ID: {default_store.id}")

## Choosing the Right Approach

| Scenario | Use |
|----------|-----|
| "I'm just trying Jockey out" | No config (default) |
| "I know the domain but not the exact fields" | Natural language description |
| "I need specific typed fields in my pipeline" | JSON Schema |
| "I'm building a demo" | Natural language description |

## Common Pitfalls

- **Too specific can hurt.** An overly narrow schema may cause Jockey to miss relevant content. Start broader, narrow later.
- **Description quality matters.** Vague descriptions produce vague extraction. Be specific about what you care about.
- **Missing property descriptions fail validation.** Every property in a JSON Schema must include a `description`. Omitting one returns a `422` error.

## Next Steps

- [Building Knowledge Stores](building_knowledge_stores.ipynb) — create stores and add videos and images
- [Querying](querying.ipynb) — ask questions about your indexed collection
- [Uploading Content](uploading_content.ipynb) — upload assets before indexing

**API Reference:** [POST /knowledge-stores](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/knowledge-stores/create-knowledge-store)